# 06 — Prompt iteration: `forward_v1` (LLM forward, Method C)

Purpose: run `forward_v1` on 20 Spider **train** examples, inspect the worst
failures against sqlglot-derived Tier-1 train gold, categorise them, and
decide whether `forward_v1` needs a targeted rule addition before it's
frozen for the dev run.

**This notebook makes real calls to the Anthropic API** (Haiku 4.5), bounded
by `cost_cap_usd=5.0`. Never run on Spider **dev** here — dev is reserved
for the final reported numbers; this notebook only ever touches train.

Constraints (locked before running):
- Exactly 20 examples, stratified by Spider hardness bucket (5 easy / 9
  medium / 3 hard / 3 extra), `seed=42`.
- Gold is sqlglot-derived Tier-1 **train** gold
  (`data/processed/gold_links_train_mentioned.json`) — Taniguchi is dev-only.
- Do not iterate on more than these 20 examples, do not peek at dev, and do
  not tune the prompt to fix specific examples ("if the schema has X, add
  Y") — only general rule changes are allowed.

In [1]:
import json
import os
import random
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))


def load_dotenv(path: Path) -> None:
    """Minimal .env loader (no python-dotenv dependency for one env var).

    Strips a single layer of matching quotes around the value — .env files
    commonly quote secrets (e.g. ANTHROPIC_API_KEY="sk-ant-...") and a naive
    split('=') would otherwise send the literal quote characters as part of
    the key, which the Anthropic API rejects with a 401.
    """
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in ("'", '"'):
            value = value[1:-1]
        os.environ.setdefault(key, value)


load_dotenv(REPO_ROOT / ".env")

from schema_linking.base import from_predictions_to_dict
from schema_linking.data_loader import load_spider_questions
from schema_linking.evaluator import evaluate
from schema_linking.llm_linker import LLMForwardLinker
from schema_linking.schema_parser import load_schemas
from schema_linking.utils.difficulty import difficulty_for_examples
from schema_linking.utils.llm_client import LLMClient
from schema_linking.utils.prompts import FORWARD_V1, FORWARD_V2, render_schema_block

SEED = 42
BUCKET_COUNTS = {"easy": 5, "medium": 9, "hard": 3, "extra": 3}
LOG_PATH = REPO_ROOT / "outputs" / "logs" / "llm_calls_prompt_iteration.jsonl"
SELECTION_PATH = REPO_ROOT / "data" / "processed" / "prompt_iteration_set.json"
FEWSHOT_PATH = REPO_ROOT / "data" / "processed" / "few_shot_examples.json"
GOLD_TRAIN_TIER1_PATH = REPO_ROOT / "data" / "processed" / "gold_links_train_mentioned.json"

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 160)

## 1. Load train examples, schemas, sqlglot Tier-1 train gold, hardness

In [2]:
train_examples = list(load_spider_questions("train"))
schemas = load_schemas()
gold_raw = json.load(GOLD_TRAIN_TIER1_PATH.open())
gold_train = {int(qid): entry for qid, entry in gold_raw.items()}
hardness = difficulty_for_examples(train_examples)

print(f"{len(train_examples)} train examples, {len(schemas)} schemas, {len(gold_train)} Tier-1 gold entries")

7000 train examples, 166 schemas, 7000 Tier-1 gold entries


## 2. Load few-shot examples and enrich with rendered schema blocks

`LLMForwardLinker` expects each few-shot dict to already carry its own
`schema_block` (see `llm_linker.py` docstring) — `predict_one` only ever
sees the *current* question's schema, never the few-shot examples' schemas.

In [3]:
few_shot = json.load(FEWSHOT_PATH.open())
for ex in few_shot:
    ex["schema_block"] = render_schema_block(schemas[ex["db_id"]])

for ex in few_shot:
    print(f"{ex['pattern']:>12}: qid={ex['question_id']} db={ex['db_id']!r} — {ex['question']!r}")

      simple: qid=4913 db='store_product' — 'What is the total number of residents for the districts with the 3 largest areas?'
 multi_table: qid=334 db='product_catalog' — 'Which attribute definitions have attribute value 0? Give me the attribute name and attribute ID.'


## 3. Stratified sample: 20 train examples, seed=42

5 easy / 9 medium / 3 hard / 3 extra (Spider hardness buckets), excluding
the few-shot question_ids from the candidate pool so we never test on an
example that's already baked into the prompt. Saved to
`data/processed/prompt_iteration_set.json`.

In [4]:
fewshot_qids = {ex["question_id"] for ex in few_shot}

by_bucket: dict[str, list] = {"easy": [], "medium": [], "hard": [], "extra": []}
for ex in train_examples:
    if ex.question_id in fewshot_qids or ex.question_id not in gold_train:
        continue
    by_bucket[hardness[ex.question_id]].append(ex)

rng = random.Random(SEED)
selected = []
for bucket, count in BUCKET_COUNTS.items():
    pool = list(by_bucket[bucket])
    rng.shuffle(pool)
    chosen = pool[:count]
    assert len(chosen) == count, f"bucket {bucket!r} has only {len(chosen)} candidates, need {count}"
    selected.extend(chosen)

assert len(selected) == 20
assert fewshot_qids.isdisjoint(ex.question_id for ex in selected)

selection_records = [
    {"question_id": ex.question_id, "db_id": ex.db_id, "question": ex.question, "hardness": hardness[ex.question_id]}
    for ex in selected
]
SELECTION_PATH.parent.mkdir(parents=True, exist_ok=True)
with SELECTION_PATH.open("w", encoding="utf-8") as f:
    json.dump(selection_records, f, indent=2)

print(f"Selected {len(selected)} examples, saved to {SELECTION_PATH}")
pd.DataFrame(selection_records)[["question_id", "db_id", "hardness", "question"]]

Selected 20 examples, saved to /Users/mac/Documents/Masters BHT 2023-2025/Semester 6 2026 april-sep/code/data/processed/prompt_iteration_set.json


,question_id,db_id,hardness,question
0,1550,customers_and_invoices,easy,Count the number of customers who have an account.
1,4052,student_1,easy,Find the last names of teachers teaching in classroom 109.
2,6623,driving_school,easy,What are the ids of all vehicles?
3,235,musical,easy,Count the number of actors.
4,452,allergy_1,easy,How many animal type allergies exist?
5,324,product_catalog,medium,Which catalog content has the highest height? Give me the catalog entry name.
6,963,medicine_enzyme_interaction,medium,What is the id and trade name of the medicines can interact with at least 3 enzymes?
7,3128,assets_maintenance,medium,How many assets does each third party company supply? List the count and the company id.
8,2054,party_people,medium,Which minister left office the latest?
9,5546,products_gen_characteristics,medium,"What is the color code and description of the product named ""chervil""?"


## 4. Run `forward_v1` on the 20 examples

**Real Anthropic API calls start here** — 20 examples x `k_samples=3` = 60
calls, logged to a dedicated `outputs/logs/llm_calls_prompt_iteration.jsonl`
(separate from the eventual full dev-run log, so this phase's `$5` cap is
independent of that later, larger run). Looping `predict_one` (not
`predict_all`) so each example's inspection block prints immediately.

In [5]:
llm_client_v1 = LLMClient(
    model="claude-haiku-4-5-20251001",
    temperature=0.7,  # >0 so k_samples=3 gives a real self-consistency signal
    max_tokens=1024,
    log_path=LOG_PATH,
    cost_cap_usd=5.0,
)
linker_v1 = LLMForwardLinker(
    llm_client=llm_client_v1,
    prompt=FORWARD_V1,
    few_shot=few_shot,
    k_samples=3,
    aggregation="union",
    extra_metadata={"phase": "prompt_iteration", "prompt_version": FORWARD_V1.version},
)

predictions_v1 = {}
for i, ex in enumerate(selected):
    schema = schemas[ex.db_id]
    pred = linker_v1.predict_one(ex, schema)
    predictions_v1[ex.question_id] = pred
    gold_entry = gold_train[ex.question_id]
    print(f"[{i + 1:>2}/20] qid={ex.question_id:<5} db={ex.db_id:<28} hardness={hardness[ex.question_id]:<6} "
          f"cost=${pred.extra['total_cost_usd']:.5f}  tokens_in={pred.extra['total_input_tokens']} "
          f"tokens_out={pred.extra['total_output_tokens']}")
    print(f"          Q: {ex.question}")
    print(f"          gold      : tables={gold_entry['tables']} columns={gold_entry['columns']}")
    for s_idx, sample in enumerate(pred.extra["sample_predictions"]):
        print(f"          sample {s_idx}: tables={sample['tables']} columns={sample['columns']}")
    print(f"          aggregated: tables={list(pred.tables)} columns={[list(c) for c in pred.columns]}")

print(f"\nforward_v1 total cost: ${sum(p.extra['total_cost_usd'] for p in predictions_v1.values()):.5f}")

[ 1/20] qid=1550  db=customers_and_invoices       hardness=easy   cost=$0.00799  tokens_in=7356 tokens_out=126
          Q: Count the number of customers who have an account.
          gold      : tables=['Accounts'] columns=[['Accounts', 'customer_id']]
          sample 0: tables=['Customers', 'Accounts'] columns=[['Customers', 'customer_id'], ['Accounts', 'customer_id']]
          sample 1: tables=['Customers', 'Accounts'] columns=[['Customers', 'customer_id'], ['Accounts', 'customer_id']]
          sample 2: tables=['Customers', 'Accounts'] columns=[['Customers', 'customer_id'], ['Accounts', 'customer_id']]
          aggregated: tables=['Customers', 'Accounts'] columns=[['Customers', 'customer_id'], ['Accounts', 'customer_id']]


[ 2/20] qid=4052  db=student_1                    hardness=easy   cost=$0.00378  tokens_in=3288 tokens_out=99
          Q: Find the last names of teachers teaching in classroom 109.
          gold      : tables=['teachers'] columns=[['teachers', 'Classroom'], ['teachers', 'LastName']]
          sample 0: tables=['teachers'] columns=[['teachers', 'LastName'], ['teachers', 'Classroom']]
          sample 1: tables=['teachers'] columns=[['teachers', 'LastName'], ['teachers', 'Classroom']]
          sample 2: tables=['teachers'] columns=[['teachers', 'LastName'], ['teachers', 'Classroom']]
          aggregated: tables=['teachers'] columns=[['teachers', 'LastName'], ['teachers', 'Classroom']]


[ 3/20] qid=6623  db=driving_school               hardness=easy   cost=$0.00624  tokens_in=5808 tokens_out=87
          Q: What are the ids of all vehicles?
          gold      : tables=['Vehicles'] columns=[['Vehicles', 'vehicle_id']]
          sample 0: tables=['Vehicles'] columns=[['Vehicles', 'vehicle_id']]
          sample 1: tables=['Vehicles'] columns=[['Vehicles', 'vehicle_id']]
          sample 2: tables=['Vehicles'] columns=[['Vehicles', 'vehicle_id']]
          aggregated: tables=['Vehicles'] columns=[['Vehicles', 'vehicle_id']]


[ 4/20] qid=235   db=musical                      hardness=easy   cost=$0.00398  tokens_in=3570 tokens_out=81
          Q: Count the number of actors.
          gold      : tables=['actor'] columns=[]
          sample 0: tables=['actor'] columns=[['actor', 'Actor_ID']]
          sample 1: tables=['actor'] columns=[['actor', 'Actor_ID']]
          sample 2: tables=['actor'] columns=[['actor', 'Actor_ID']]
          aggregated: tables=['actor'] columns=[['actor', 'Actor_ID']]


[ 5/20] qid=452   db=allergy_1                    hardness=easy   cost=$0.00436  tokens_in=3816 tokens_out=108
          Q: How many animal type allergies exist?
          gold      : tables=['Allergy_Type'] columns=[['Allergy_Type', 'AllergyType']]
          sample 0: tables=['Allergy_Type'] columns=[['Allergy_Type', 'AllergyType']]
          sample 1: tables=['Allergy_Type'] columns=[['Allergy_Type', 'AllergyType']]
          sample 2: tables=['Allergy_Type'] columns=[['Allergy_Type', 'AllergyType']]
          aggregated: tables=['Allergy_Type'] columns=[['Allergy_Type', 'AllergyType']]


...

[16/20] qid=4610  db=entertainment_awards         hardness=hard   cost=$0.00475  tokens_in=3846 tokens_out=180
          Q: Show the names of festivals that have nominated artworks of type "Program Talent Show".
          gold      : tables=['artwork', 'festival_detail'] columns=[['artwork', 'Type'], ['festival_detail', 'Festival_Name']]
          sample 0: tables=['festival_detail', 'artwork', 'nomination'] columns=[['festival_detail', 'Festival_Name'], ['artwork', 'Type'], ['nomination', 'Artwork_ID'], ['nomination', 'Festival_ID']]
          sample 1: tables=['festival_detail', 'artwork', 'nomination'] columns=[['festival_detail', 'Festival_Name'], ['artwork', 'Type'], ['nomination', 'Artwork_ID'], ['nomination', 'Festival_ID']]
          sample 2: tables=['festival_detail', 'artwork', 'nomination'] columns=[['festival_detail', 'Festival_Name'], ['artwork', 'Type'], ['nomination', 'Festival_ID'], ['nomination', 'Artwork_ID']]
          aggregated: tables=['festival_detail', 'artwork

[17/20] qid=3385  db=sports_competition           hardness=hard   cost=$0.00555  tokens_in=4743 tokens_out=162
          Q: which countries did participated in both Friendly and Tournament type competitions.
          gold      : tables=['competition'] columns=[['competition', 'Competition_type'], ['competition', 'Country']]
          sample 0: tables=['competition'] columns=[['competition', 'Country'], ['competition', 'Competition_type']]
          sample 1: tables=['competition'] columns=[['competition', 'Country'], ['competition', 'Competition_type']]
          sample 2: tables=['competition', 'competition_result', 'club'] columns=[['competition', 'Country'], ['competition', 'Competition_type'], ['competition_result', 'Club_ID_1'], ['competition_result', 'Club_ID_2'], ['competition_result', 'Competition_ID']]
          aggregated: tables=['competition', 'competition_result', 'club'] columns=[['competition', 'Country'], ['competition', 'Competition_type'], ['competition_result', 'Clu

[18/20] qid=5877  db=tracking_share_transactions  hardness=extra  cost=$0.00609  tokens_in=5166 tokens_out=184
          Q: Show the description of the transaction type that occurs most frequently.
          gold      : tables=['Ref_Transaction_Types'] columns=[['Ref_Transaction_Types', 'transaction_type_code'], ['Ref_Transaction_Types', 'transaction_type_description']]
          sample 0: tables=['Ref_Transaction_Types', 'Transactions'] columns=[['Ref_Transaction_Types', 'transaction_type_description'], ['Transactions', 'transaction_type_code']]
          sample 1: tables=['Transactions', 'Ref_Transaction_Types'] columns=[['Transactions', 'transaction_type_code'], ['Ref_Transaction_Types', 'transaction_type_code'], ['Ref_Transaction_Types', 'transaction_type_description']]
          sample 2: tables=['Ref_Transaction_Types', 'Transactions'] columns=[['Ref_Transaction_Types', 'transaction_type_description'], ['Transactions', 'transaction_type_code']]
          aggregated: tables=['Ref_

[19/20] qid=5794  db=customer_complaints          hardness=extra  cost=$0.00605  tokens_in=4938 tokens_out=222
          Q: Find the last name of the staff member who processed the complaint of the cheapest product.
          gold      : tables=['Products', 'Staff'] columns=[['Products', 'product_price'], ['Staff', 'last_name']]
          sample 0: tables=['Staff', 'Complaints', 'Products'] columns=[['Staff', 'last_name'], ['Complaints', 'staff_id'], ['Complaints', 'product_id'], ['Products', 'product_price']]
          sample 1: tables=['Staff', 'Complaints', 'Products'] columns=[['Staff', 'last_name'], ['Complaints', 'staff_id'], ['Complaints', 'product_id'], ['Products', 'product_id'], ['Products', 'product_price']]
          sample 2: tables=['Staff', 'Complaints', 'Products'] columns=[['Staff', 'last_name'], ['Complaints', 'staff_id'], ['Complaints', 'product_id'], ['Products', 'product_id'], ['Products', 'product_price']]
          aggregated: tables=['Staff', 'Complaints', 'Prod

[20/20] qid=3641  db=baseball_1                   hardness=extra  cost=$0.01983  tokens_in=19053 tokens_out=156
          Q: In 2014, what are the id and rank of the team that has the largest average number of attendance?
          gold      : tables=['home_game', 'team'] columns=[['home_game', 'attendance'], ['home_game', 'team_id'], ['home_game', 'year'], ['team', 'rank'], ['team', 'team_id']]
          sample 0: tables=['team'] columns=[['team', 'year'], ['team', 'team_id'], ['team', 'rank'], ['team', 'attendance']]
          sample 1: tables=['team'] columns=[['team', 'year'], ['team', 'team_id'], ['team', 'rank'], ['team', 'attendance']]
          sample 2: tables=['team'] columns=[['team', 'year'], ['team', 'team_id'], ['team', 'rank'], ['team', 'attendance']]
          aggregated: tables=['team'] columns=[['team', 'year'], ['team', 'team_id'], ['team', 'rank'], ['team', 'attendance']]

forward_v1 total cost: $0.12459


## 5. Per-query F1 vs sqlglot Tier-1 train gold

`gold` is subsetted to exactly these 20 `question_id`s — `evaluate()`
iterates `gold.keys()` as its driving set, so passing the full 7000-entry
train gold would silently evaluate all of train instead of just the sample.
`mean_f1 = (table_f1 + column_f1) / 2`, matching this project's existing
selection-rule convention (see the embedding-tuning notebook).

In [6]:
def per_query_table(predictions: dict, method_name: str) -> pd.DataFrame:
    pred_dict = from_predictions_to_dict(predictions)
    gold_subset = {qid: gold_train[qid] for qid in pred_dict}
    result = evaluate(
        predictions=pred_dict, gold=gold_subset, schemas=schemas, hardness=hardness,
        method_name=method_name, tier_name="tier1",
    )
    per_query = result.per_query.copy()
    per_query["mean_f1"] = (per_query["table_f1"] + per_query["column_f1"]) / 2
    return per_query.sort_values("mean_f1").reset_index(drop=True)


per_query_v1 = per_query_table(predictions_v1, "llm_forward_v1")
per_query_v1[["question_id", "db_id", "hardness", "table_f1", "column_f1", "mean_f1",
              "hallucinated_tables_list", "hallucinated_columns_list"]]

,question_id,db_id,hardness,table_f1,column_f1,mean_f1,hallucinated_tables_list,hallucinated_columns_list
0,235,musical,easy,1.000000,0.000000,0.500000,,
1,3385,sports_competition,hard,0.500000,0.571429,0.535714,,
2,3641,baseball_1,extra,0.666667,0.444444,0.555556,,
3,963,medicine_enzyme_interaction,medium,0.666667,0.666667,0.666667,,
4,1550,customers_and_invoices,easy,0.666667,0.666667,0.666667,,
5,3128,assets_maintenance,medium,0.666667,0.666667,0.666667,,
6,5794,customer_complaints,extra,0.800000,0.571429,0.685714,,
7,5877,tracking_share_transactions,extra,0.666667,0.800000,0.733333,,
8,4610,entertainment_awards,hard,0.800000,0.666667,0.733333,,
9,5546,products_gen_characteristics,medium,1.000000,0.666667,0.833333,,


## 6. Worst 5 cases — full detail

In [7]:
worst5_qids = per_query_v1["question_id"].head(5).tolist()

for qid in worst5_qids:
    ex = next(e for e in selected if e.question_id == qid)
    gold_entry = gold_train[qid]
    pred = predictions_v1[qid]
    row = per_query_v1[per_query_v1["question_id"] == qid].iloc[0]
    print("=" * 100)
    print(f"qid={qid} db={ex.db_id} hardness={hardness[qid]}")
    print(f"Question: {ex.question}")
    print(f"Gold SQL: {ex.query}")
    print(f"Gold tables: {gold_entry['tables']}")
    print(f"Gold columns: {gold_entry['columns']}")
    print(f"table_f1={row['table_f1']:.3f} column_f1={row['column_f1']:.3f}")
    print(f"Hallucinated tables: {row['hallucinated_tables_list']!r}  columns: {row['hallucinated_columns_list']!r}")
    print(f"n_samples_parsed={pred.extra['n_samples_parsed']} n_samples_valid={pred.extra['n_samples_valid']}")
    print("Raw samples:")
    for i, s in enumerate(pred.extra["sample_predictions"]):
        print(f"  sample {i}: tables={s['tables']} columns={s['columns']}")
    print(f"Aggregated (union): tables={list(pred.tables)} columns={[list(c) for c in pred.columns]}")

qid=235 db=musical hardness=easy
Question: Count the number of actors.
Gold SQL: SELECT count(*) FROM actor
Gold tables: ['actor']
Gold columns: []
table_f1=1.000 column_f1=0.000
Hallucinated tables: ''  columns: ''
n_samples_parsed=3 n_samples_valid=3
Raw samples:
  sample 0: tables=['actor'] columns=[['actor', 'Actor_ID']]
  sample 1: tables=['actor'] columns=[['actor', 'Actor_ID']]
  sample 2: tables=['actor'] columns=[['actor', 'Actor_ID']]
Aggregated (union): tables=['actor'] columns=[['actor', 'Actor_ID']]
qid=3385 db=sports_competition hardness=hard
Question: which countries did participated in both Friendly and Tournament type competitions.
Gold SQL: SELECT country FROM competition WHERE competition_type  =  'Friendly' INTERSECT SELECT country FROM competition WHERE competition_type  =  'Tournament'
Gold tables: ['competition']
Gold columns: [['competition', 'Competition_type'], ['competition', 'Country']]
table_f1=0.500 column_f1=0.571
Hallucinated tables: ''  columns: ''
n_sa

## 7. Failure categorisation (manual)

Categorising each of the worst 5 by hand, against the fixed vocabulary
(`Hallucination`, `Missing an obvious element`, `Column-only, forgot table`,
`Included join-only column`, `JSON parse error`, `Other`):

1. **qid=235 (musical)** — `SELECT count(*) FROM actor`. Gold has table
   `actor` and **no columns at all** (nothing is selected beyond the count).
   All 3 samples added `actor.Actor_ID` anyway — a fabricated "representative"
   column for a bare `COUNT(*)`. Not a hallucination (the column is real),
   not join-only (no join involved), not "forgot table" (table is correct).
   Category: **Other** — the model treats `COUNT(*)` as if it needs some
   column to point at, despite rule 3 and the `simple` few-shot example
   (`"How many singers are there?"` -> `columns: []`) both already modelling
   the opposite.

2. **qid=3385 (sports_competition)** — gold SQL is a same-table `INTERSECT`
   (`SELECT country FROM competition WHERE ... INTERSECT SELECT country
   FROM competition WHERE ...`) — **only** the `competition` table is ever
   touched. All 3 samples nonetheless added `competition_result` and `club`
   plus their join-key columns (`Club_ID_1`, `Club_ID_2`, `Competition_ID`),
   as if answering the question required joining to a results table.
   Category: **Included join-only column** (extended to whole tables — the
   added tables/columns exist only to support a join that the actual gold
   SQL never performs).

3. **qid=3641 (baseball_1)** — gold SQL joins `home_game` and `team`;
   `home_game` supplies `attendance`/`year`, `team` supplies `rank`. All 3
   samples predicted `team` only, attributing `attendance`/`year` to it
   instead of the (entirely missing) `home_game` table. Category:
   **Missing an obvious element** — `home_game` never appears in any of the
   3 samples. This looks like a genuine reasoning/domain gap (not realising
   "average attendance" implies a separate per-game record), not a prompt-
   wording issue — nothing in the rules currently misleads the model here.

4. **qid=963 (medicine_enzyme_interaction)** — gold SQL joins `medicine` to
   `medicine_enzyme_interaction` only to `HAVING COUNT(*) >= 3`; the bridge
   table contributes no output column. Gold is `medicine` only. All 3
   samples added `medicine_enzyme_interaction` plus its two FK columns.
   Category: **Included join-only column** — same pattern as qid=3385, this
   time a real join-bridge table rather than an unused side table.

5. **qid=1550 (customers_and_invoices)** — gold SQL is
   `SELECT count(DISTINCT customer_id) FROM Accounts` — **no join at all**,
   `Customers` is never referenced. All 3 samples added `Customers` anyway,
   presumably because the question's word "customers" is topically salient.
   Category: **Included join-only column** (broadly construed — the shared
   failure mode across 2/3/4/5 is "added a table/column that's schema-real
   and plausible-sounding but not actually referenced by the gold SQL",
   whether or not a join is literally involved).

## 8. STOP — tally and decision

| Category | Count |
| --- | --- |
| Included join-only column | 3 (qid 3385, 963, 1550) |
| Missing an obvious element | 1 (qid 3641) |
| Other | 1 (qid 235) |

**Decision rule:** draft `forward_v2` only if ≥3 of the worst 5 share one
category *and* that category is a fixable prompt issue, not a fundamental
limitation.

**3 of 5 (60%) share "Included join-only column"**, meeting the threshold.
Is it fixable in the prompt? Yes — rule 3 in `forward_v1` already targets
exactly this failure mode, but only for *columns* ("Do NOT return columns
whose ONLY purpose is joining tables together"). It says nothing about
whole *tables* being over-included for the same reason, and nothing about
tables that are topically plausible but never actually referenced by the
SQL at all (qid=1550's `Customers` isn't even a join partner — it's simply
unused). Generalising rule 3 from "columns" to "tables and columns", and
from "join-only" to "must correspond to something the SQL actually
references", is a general rule change — not a per-example patch — so it's
in scope.

**-> Drafting `forward_v2`.** (qid=235's "Other" and qid=3641's "Missing an
obvious element" are each single occurrences and don't meet the ≥3
threshold on their own; qid=3641 in particular looks like a genuine model
reasoning gap the prompt can't generally fix without becoming schema-
specific, which is explicitly out of bounds.)

## 9. `forward_v2` — targeted rule addition

See `src/schema_linking/utils/prompts.py` for the actual registered
`PromptTemplate`. Only rule 3 changes from v1; rules 1, 2, 4 and the user
template / output schema are byte-identical.

In [8]:
print("--- forward_v1 system ---")
print(FORWARD_V1.system)
print()
print("--- forward_v2 system ---")
print(FORWARD_V2.system)

--- forward_v1 system ---
You extract the tables and columns from a database schema that are relevant to answering a natural-language question.

Rules:
1. Use ONLY tables and columns that appear in the provided schema.
2. Return every table you need. Return every column you need.
3. Do NOT return columns whose ONLY purpose is joining tables together — return only columns that would appear in SELECT, WHERE, GROUP BY, HAVING, or ORDER BY of the target SQL.
4. Output ONLY a JSON object with keys "tables" (list of strings) and "columns" (list of [table_name, column_name] pairs). No prose, no explanation, no markdown fences.

--- forward_v2 system ---
You extract the tables and columns from a database schema that are relevant to answering a natural-language question.

Rules:
1. Use ONLY tables and columns that appear in the provided schema.
2. Return every table you need. Return every column you need.
3. A table or column must correspond to something the target SQL actually references (in S

## 10. Re-run the same 20 examples with `forward_v2`

Same examples, same few-shot, same model/temperature/k_samples — only the
prompt version differs. Same dedicated log file, tagged
`prompt_version=forward_v2` for traceability.

In [9]:
llm_client_v2 = LLMClient(
    model="claude-haiku-4-5-20251001",
    temperature=0.7,
    max_tokens=1024,
    log_path=LOG_PATH,
    cost_cap_usd=5.0,
)
linker_v2 = LLMForwardLinker(
    llm_client=llm_client_v2,
    prompt=FORWARD_V2,
    few_shot=few_shot,
    k_samples=3,
    aggregation="union",
    extra_metadata={"phase": "prompt_iteration", "prompt_version": FORWARD_V2.version},
)

predictions_v2 = {}
for i, ex in enumerate(selected):
    schema = schemas[ex.db_id]
    pred = linker_v2.predict_one(ex, schema)
    predictions_v2[ex.question_id] = pred
    print(f"[{i + 1:>2}/20] qid={ex.question_id:<5} db={ex.db_id:<28} "
          f"cost=${pred.extra['total_cost_usd']:.5f}  aggregated_tables={list(pred.tables)}")

print(f"\nforward_v2 total cost: ${sum(p.extra['total_cost_usd'] for p in predictions_v2.values()):.5f}")

[ 1/20] qid=1550  db=customers_and_invoices       cost=$0.00816  aggregated_tables=['Customers', 'Accounts']


[ 2/20] qid=4052  db=student_1                    cost=$0.00388  aggregated_tables=['teachers']


[ 3/20] qid=6623  db=driving_school               cost=$0.00634  aggregated_tables=['Vehicles']


[ 4/20] qid=235   db=musical                      cost=$0.00408  aggregated_tables=['actor']


[ 5/20] qid=452   db=allergy_1                    cost=$0.00449  aggregated_tables=['Allergy_Type']


...

[16/20] qid=4610  db=entertainment_awards         cost=$0.00485  aggregated_tables=['festival_detail', 'artwork', 'nomination']


[17/20] qid=3385  db=sports_competition           cost=$0.00554  aggregated_tables=['competition', 'competition_result']


[18/20] qid=5877  db=tracking_share_transactions  cost=$0.00630  aggregated_tables=['Transactions', 'Ref_Transaction_Types']


[19/20] qid=5794  db=customer_complaints          cost=$0.00624  aggregated_tables=['Staff', 'Complaints', 'Products']


[20/20] qid=3641  db=baseball_1                   cost=$0.01994  aggregated_tables=['team']

forward_v2 total cost: $0.12742


In [10]:
per_query_v2 = per_query_table(predictions_v2, "llm_forward_v2")
per_query_v2[["question_id", "db_id", "hardness", "table_f1", "column_f1", "mean_f1",
              "hallucinated_tables_list", "hallucinated_columns_list"]]

,question_id,db_id,hardness,table_f1,column_f1,mean_f1,hallucinated_tables_list,hallucinated_columns_list
0,235,musical,easy,1.000000,0.000000,0.500000,,
1,3641,baseball_1,extra,0.666667,0.444444,0.555556,,
2,5794,customer_complaints,extra,0.800000,0.500000,0.650000,,
3,963,medicine_enzyme_interaction,medium,0.666667,0.666667,0.666667,,
4,1550,customers_and_invoices,easy,0.666667,0.666667,0.666667,,
5,3128,assets_maintenance,medium,0.666667,0.666667,0.666667,,
6,2891,icfp_1,hard,0.800000,0.600000,0.700000,,
7,4610,entertainment_awards,hard,0.800000,0.666667,0.733333,,
8,5877,tracking_share_transactions,extra,0.666667,0.800000,0.733333,,
9,3385,sports_competition,hard,0.666667,0.800000,0.733333,,


In [11]:
comparison = pd.DataFrame({
    "v1_mean_table_f1": [per_query_v1["table_f1"].mean()],
    "v1_mean_column_f1": [per_query_v1["column_f1"].mean()],
    "v1_mean_f1": [per_query_v1["mean_f1"].mean()],
    "v2_mean_table_f1": [per_query_v2["table_f1"].mean()],
    "v2_mean_column_f1": [per_query_v2["column_f1"].mean()],
    "v2_mean_f1": [per_query_v2["mean_f1"].mean()],
})
comparison

,v1_mean_table_f1,v1_mean_column_f1,v1_mean_f1,v2_mean_table_f1,v2_mean_column_f1,v2_mean_f1
0,0.861667,0.786032,0.823849,0.87,0.766746,0.818373


## 11. Comparison and final decision

This run: `v1_mean_f1 = 0.8238` vs `v2_mean_f1 = 0.8184` (table above) — `v1`
is marginally ahead (~0.006, less than the weight of a single query
changing category out of 20).

**This margin is not reliable.** Across the runs made while building this
notebook, the v1-vs-v2 ranking **flipped sign three times**: an early
exploratory pass had v1 ahead by ~0.007, the first full execution of this
notebook had v2 ahead by ~0.008, and this final execution has v1 ahead by
~0.006. All three deltas are of the same small magnitude and inconsistent
in direction — the honest read is that `k_samples=3` at `temperature=0.7`
over only 20 examples cannot distinguish these two prompts' aggregate
quality; the true effect (if any) is smaller than this iteration set's
noise floor.

What *did* reproduce consistently across every run: the specific worst-5
qids, their per-query F1s (up to ~0.03 noise on one query), and — most
importantly — **`forward_v2`'s rule-3 rewrite did not measurably fix the
"included join-only column/table" pattern** on the cases that motivated it
(qid=3385 still pulls in `competition_result`; qid=963 still pulls in
`medicine_enzyme_interaction`; qid=1550 still pulls in `Customers`, in
every run including this one — see the `aggregated_tables` printed above).
The rewritten rule is *more precise* in wording but evidently not more
*effective* at steering the model away from this behaviour.

**Decision: lock `forward_v1`.** `forward_v2` was a well-motivated, general
(non-schema-specific) rule change, tried exactly as the process requires —
but it produced no reproducible improvement, and simplicity wins a tie.
`forward_v2` stays registered in `prompts.py` (useful for a future ablation
with more samples or higher `k_samples`) but is not the frozen version.